# IntroDS M2 — FINAL COMPLETE NOTEBOOK

**Một notebook thực sự**, toàn bộ runner 688 dòng tách theo cell rõ ràng. Parser Rev10 được nhúng sẵn trong file này (không cần .py hoặc ZIP phụ).

Quy trình: thư viện → Drive → cấu hình → engine → parser + QA → preflight → sửa/merge range 1–2 → chạy 3–5 → master → báo cáo.

Dữ liệu v5/v6 được giữ nguyên. Chỉ các bài cải thiện được thay vào bản output riêng. Master **PROVISIONAL** cho đến khi semantic review hoàn tất.


## 1. Cài thư viện


In [ ]:
%pip -q install psutil 'bibtexparser>=1.4,<2' google-api-python-client pytest


## 2. Kết nối Google Drive


In [ ]:
from google.colab import drive, auth
drive.mount('/content/drive', force_remount=False)
auth.authenticate_user()


## 3. Cấu hình: chọn range và output
Mặc định `REPAIR_RANGES=[1,2]` và `NEW_RANGES=[3,4,5]`. Có thể chia nhau chạy từng range. Không ghi đè v6.


In [ ]:
import os
import sys
import json
import re
import time
import zipfile
import shutil
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter


# ======================== USER CONFIGURATION ========================
M1_PARENT_ID = "AUTO"
V6_FOLDER_ID = "AUTO"
BASE = Path("/content/drive/MyDrive/PROJECT HCMUS/PROJECT INTRODS")
CODE_ZIP = Path("/content/m2_final_bundled_rev10.zip")
OUT = BASE / "M2_processed_FINAL_onepass"
LOCAL = Path("/content/m2_final_onepass_workspace")
PROJECT = Path("/content/m2_final_onepass_code")
REPAIR_RANGES = [1, 2]        # Set [] to skip selective repair
NEW_RANGES = [3, 4, 5]        # Can run [3] then [4,5] in a later session
MODE = "heading_plus_partial"
WORKERS = 1
FULL_QA_REPAIR = True         # Full JSONL/hierarchy rescan after any replacement
FULL_QA_NEW = True
CREATE_MASTER = True         # Combined ZIP is explicitly PROVISIONAL pending semantic review
SKIP_VERIFIED_FINISHED = True
# ====================================================================


print('Repair:',REPAIR_RANGES,'New:',NEW_RANGES,'Output:',OUT)


## 4. Engine Rev10 tích hợp trong notebook
Cell này dài vì chứa ZIP engine ở dạng base64. Có thể thu gọn cell.


In [ ]:
# Bundled Rev10 engine (no extra ZIP or .py required).
import io, base64
_ZIP64=(
  "..."
)
_b=base64.b64decode(_ZIP64,validate=True)
assert len(_b)==284838
with zipfile.ZipFile(io.BytesIO(_b)) as _zf:
  assert _zf.testzip() is None
CODE_ZIP.write_bytes(_b)
print(f'Embedded parser engine ready: {len(_b):,} bytes')
del _ZIP64,_b


## 5. Tiện ích và preflight


In [ ]:
def now():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def log(message):
    print(f"[{now()}] {message}", flush=True)


def write_json(path, payload):
    Path(path).write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for part in iter(lambda: f.read(4 * 1024 * 1024), b""):
            h.update(part)
    return h.hexdigest()


def preflight():
    assert REPAIR_RANGES or NEW_RANGES, "No ranges selected."
    assert set(REPAIR_RANGES).issubset({1, 2})
    assert set(NEW_RANGES).issubset({3, 4, 5})
    assert len(set(REPAIR_RANGES)) == len(REPAIR_RANGES)
    assert len(set(NEW_RANGES)) == len(NEW_RANGES)
    assert MODE in {"heading", "heading_plus_partial", "semantic_all"}
    assert WORKERS >= 1
    from google.colab import drive, auth
    drive.mount("/content/drive", force_remount=False)
    auth.authenticate_user()
    assert CODE_ZIP.is_file(), f"Missing code ZIP: {CODE_ZIP}"
    if OUT.resolve() in [BASE.resolve(), (BASE / "M2_processed_v5").resolve(),
                         (BASE / "M2_processed_v6").resolve()]:
        raise ValueError("Refusing to overwrite baseline output.")
    OUT.mkdir(parents=True, exist_ok=True)
    PROJECT.mkdir(parents=True, exist_ok=True)
    LOCAL.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(CODE_ZIP) as z:
        wanted = {"src/m2_parsing/colab_targeted_repair.py",
                  "src/m2_parsing/selective_repair.py",
                  "src/m2_parsing/batch_pipeline.py",
                  "src/m2_parsing/hierarchy.py"}
        assert wanted.issubset(z.namelist()) and z.testzip() is None, "Invalid Rev10 ZIP"
        if any(Path(x).is_absolute() or ".." in Path(x).parts for x in z.namelist()):
            raise ValueError("Unsafe file path in code ZIP.")
        z.extractall(PROJECT)
    for mod in list(sys.modules):
        if mod == "src" or mod.startswith("src."):
            del sys.modules[mod]
    if str(PROJECT) in sys.path:
        sys.path.remove(str(PROJECT))
    sys.path.insert(0, str(PROJECT))
    from src.m2_parsing.batch_pipeline import PIPELINE_REVISION
    assert PIPELINE_REVISION == 10, f"Wrong base revision: {PIPELINE_REVISION}"
    log("Running Rev10 original regression tests before overlay...")
    subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], cwd=PROJECT, check=True)
    install_conservative_patches()
    for mod in list(sys.modules):
        if mod == "src" or mod.startswith("src."):
            del sys.modules[mod]
    test_overlay()
    log("Running M2 regression tests again AFTER the overlay...")
    subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], cwd=PROJECT, check=True)
    log("Tests and patch preflight passed. Existing v5/v6 will NOT be changed.")


## 6. Vá LaTeX wrapper và regression tests
Sửa heading bên trong `hypertarget/newtext/textcolor/colorbox/mbox/hbox` theo hướng bảo thủ; tránh macro placeholder `#1`.


In [ ]:
def install_conservative_patches():
    """Append-only compatibility fix; source ZIP remains untouched."""
    hierarchy = PROJECT / "src/m2_parsing/hierarchy.py"
    marker = "# M2_FINAL_WRAPPER_OVERLAY_20261115"
    source = hierarchy.read_text(encoding="utf-8")
    if marker not in source:
        overlay = r'''


# M2_FINAL_WRAPPER_OVERLAY_20261115
import re as _m2_final_re
_m2_final_wrapper = _m2_final_re.compile(
    r"\\(hypertarget|newtext|textcolor|colorbox|mbox|hbox)(?![A-Za-z@])")
_m2_final_heading = _m2_final_re.compile(
    r"\\(?:part|chapter|section|subsection|subsubsection|paragraph|subparagraph)"
    r"\*?\s*(?:\[[^\]\n]*\]\s*)?\{")


def _m2_final_group(src, pos, opener="{", closer="}"):
    while pos < len(src) and src[pos].isspace():
        pos += 1
    if pos >= len(src) or src[pos] != opener:
        return None
    depth = 0
    for j in range(pos, len(src)):
        c = src[j]
        if c not in (opener, closer):
            continue
        k, slashes = j - 1, 0
        while k >= 0 and src[k] == "\\":
            slashes += 1
            k -= 1
        if slashes % 2:
            continue
        if c == opener:
            depth += 1
        else:
            depth -= 1
            if depth == 0:
                return src[pos + 1:j], j + 1
    return None


def _m2_final_unwrap(tex):
    # Never rewrite a normal LaTeX preamble or macro definition body.
    marker = r"\begin{document}"
    begin = tex.find(marker)
    offset = begin + len(marker) if begin >= 0 else 0
    prefix, body = tex[:offset], tex[offset:]
    for _ in range(4):
        cursor, rendered, changed = 0, [], False
        for match in _m2_final_wrapper.finditer(body):
            if match.start() < cursor:
                continue
            kind, pos = match.group(1), match.end()
            if pos < len(body) and body[pos:].lstrip().startswith("["):
                opt_pos = pos + len(body[pos:]) - len(body[pos:].lstrip())
                optional = _m2_final_group(body, opt_pos, "[", "]")
                if optional is None:
                    continue
                pos = optional[1]
            arity = 2 if kind in {"hypertarget", "textcolor", "colorbox"} else 1
            groups = []
            for _group_no in range(arity):
                parsed = _m2_final_group(body, pos)
                if parsed is None:
                    break
                groups.append(parsed[0])
                pos = parsed[1]
            if len(groups) != arity:
                continue
            content = groups[-1]
            if not _m2_final_heading.search(content):
                continue
            if _m2_final_re.search(r"#[1-9]", content):
                continue  # macro template, NOT a live heading
            rendered.append(body[cursor:match.start()])
            rendered.append(content)
            cursor = pos
            changed = True
        if not changed:
            break
        rendered.append(body[cursor:])
        body = "".join(rendered)
    return prefix + body


_m2_final_original_parse_tex_to_tree = parse_tex_to_tree
def parse_tex_to_tree(tex):
    return _m2_final_original_parse_tex_to_tree(_m2_final_unwrap(tex))
'''
        hierarchy.write_text(source + overlay, encoding="utf-8")
    retry = PROJECT / "src/m2_parsing/selective_repair.py"
    text = retry.read_text(encoding="utf-8")
    old = "if old.get('pipeline_revision') != 9:"
    if old in text:
        text = text.replace(old, "if old.get('pipeline_revision') not in (9, 10):", 1)
    assert "if old.get('pipeline_revision') not in (9, 10):" in text
    text = text.replace("merged['base_pipeline_revision'] = 9",
                        "merged['base_pipeline_revision'] = old['pipeline_revision']")
    retry.write_text(text, encoding="utf-8")


def test_overlay():
    from src.m2_parsing.hierarchy import parse_tex_to_tree, _m2_final_unwrap
    from src.m2_parsing.selective_repair import can_replace
    samples = [
        (r"\begin{document}\hypertarget{anchor}{\section{Section Alpha} Text}\end{document}",
         r"\section{Section Alpha}"),
        (r"\begin{document}\newtext{\subsection{Section Beta} Text}\end{document}",
         r"\subsection{Section Beta}"),
        (r"\begin{document}\textcolor{blue}{\section{Section Gamma}} \end{document}",
         r"\section{Section Gamma}"),
    ]
    for src, required in samples:
        fixed = _m2_final_unwrap(src)
        assert required in fixed, f"Wrapper conversion failed: {required}"
        parse_tex_to_tree(fixed)
    macro = r"\begin{document}\newtext{\section{#1}} end\end{document}"
    assert _m2_final_unwrap(macro) == macro, "Unsafe macro definition conversion"
    log("Conservative structural-wrapper smoke tests PASS.")


## 7. Tải M1/v6, triage warning và kiểm tra SHA/CRC


In [ ]:
def drive_client():
    import google.auth
    from googleapiclient.discovery import build
    credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive.readonly"])
    return build("drive", "v3", credentials=credentials, cache_discovery=False)


def list_children(service, folder_id):
    files, token = [], None
    while True:
        page = service.files().list(
            q=f"'{folder_id}' in parents and trashed=false",
            fields="nextPageToken, files(id,name,mimeType,size)",
            pageSize=1000, pageToken=token, supportsAllDrives=True,
            includeItemsFromAllDrives=True).execute()
        files.extend(page.get("files", []))
        token = page.get("nextPageToken")
        if not token:
            return files


def folder_map(service, folder_id):
    return {x["name"]: x["id"] for x in list_children(service, folder_id)
            if x["mimeType"] == "application/vnd.google-apps.folder"}


def download(service, remote, target):
    from googleapiclient.http import MediaIoBaseDownload
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    size = int(remote["size"]) if remote.get("size") else None
    if target.exists() and size is not None and target.stat().st_size == size:
        return
    pending = target.with_name(target.name + ".downloading")
    for attempt in range(4):
        try:
            pending.unlink(missing_ok=True)
            request = service.files().get_media(fileId=remote["id"], supportsAllDrives=True)
            with pending.open("wb") as stream:
                transfer = MediaIoBaseDownload(stream, request, chunksize=8 * 1024 * 1024)
                done = False
                while not done:
                    _, done = transfer.next_chunk(num_retries=3)
            if size is not None and pending.stat().st_size != size:
                raise IOError(f"Download size mismatch: {remote['name']}")
            os.replace(pending, target)
            return
        except Exception:
            if attempt == 3:
                raise
            time.sleep(2 ** attempt)


def triage_coverage(coverage):
    """Evidence-tiered warnings; raw QA records are preserved, never deleted.
    'LIKELY' is not 'proven'. Only source-missing errors are upstream blockers.
    """
    technical_include = {"xy", "xypic", "epsf", "glyphtounicode",
                         "supp-pdf", "pdftexcmds", "epstopdf", "color",
                         "texnansi", "pdfsync"}
    rows, counts = [], Counter()
    for paper in coverage:
        pid = paper["arxiv_id"]
        for trace in paper.get("source_trace", []):
            root = trace.get("root_file") or ""
            headings = trace.get("unrepresented_headings", []) or []
            missing = trace.get("missing_includes", []) or []
            reasons, risk = [], "NONE"
            if trace.get("reason") == "missing sources directory":
                risk = "UPSTREAM_SOURCE_MISSING"
                reasons.append("M1 sources directory unavailable")
            if headings:
                if all(re.search(r"#[1-9]", h) for h in headings):
                    reasons.append("heading looks like macro placeholder (needs source check)")
                    risk = "LIKELY_FALSE_POSITIVE_MACRO" if risk == "NONE" else risk
                else:
                    reasons.append(f"{len(headings)} structural headings not represented")
                    risk = "HIGH_HEADING_REVIEW"
            if missing:
                names = [m.split(":", 1)[-1].strip().replace("\\", "/").split("/")[-1]
                         .removesuffix(".tex").lower() for m in missing]
                if all(x in technical_include for x in names):
                    reasons.append("all unresolved inputs look like TeX support files (not proven harmless)")
                    if risk == "NONE":
                        risk = "LIKELY_SUPPORT_INCLUDE"
                else:
                    reasons.append("include unresolved; may hide article content")
                    if risk == "NONE":
                        risk = "MEDIUM_INCLUDE_REVIEW"
            if root and any(w in Path(root).stem.lower()
                            for w in ("template", "sample", "example", "guide")):
                reasons.append("root filename heuristic only; NOT evidence of wrong paper")
                if risk == "NONE":
                    risk = "LOW_FILENAME_ONLY"
            if trace.get("source_characters", 0) and trace.get("source_characters", 0) < 250:
                reasons.append("short TeX source")
                if risk == "NONE":
                    risk = "MEDIUM_SHORT_SOURCE"
            if not reasons:
                continue
            counts[risk] += 1
            rows.append({"arxiv_id": pid, "version": trace.get("version"),
                         "risk": risk, "evidence": reasons, "root_file": root,
                         "unrepresented_headings": headings, "missing_includes": missing,
                         "source_characters": trace.get("source_characters", 0)})
    counts["TOTAL_REVIEW_RECORDS"] = len(rows)
    counts["classification_policy"] = ("Filename-only is low risk. Macro placeholders and "
                                      "technical includes are LIKELY benign, never auto-cleared. "
                                      "All true heading discrepancies still require source inspection.")
    return rows, dict(counts)


def emit_triage(tables_zip, directory):
    from src.m2_parsing.selective_repair import read_metrics
    m = read_metrics(tables_zip)
    rows, summary = triage_coverage(m["coverage"])
    directory = Path(directory)
    with (directory / "triage_semantic.jsonl").open("w", encoding="utf-8") as stream:
        for row in rows:
            stream.write(json.dumps(row, ensure_ascii=False) + "\n")
    write_json(directory / "triage_summary.json", summary)
    return summary


def stage_publish(source_dir, target_dir):
    source_dir, target_dir = Path(source_dir), Path(target_dir)
    if target_dir.exists():
        raise FileExistsError(f"Refusing to overwrite: {target_dir}")
    pending = target_dir.parent / ("." + target_dir.name + "_upload_incomplete")
    if pending.exists():
        raise FileExistsError(f"Interrupted staging exists: {pending}; inspect before retry.")
    pending.mkdir(parents=True)
    try:
        for src in sorted(source_dir.iterdir()):
            if not src.is_file():
                continue
            dst = pending / src.name
            shutil.copy2(src, dst)
            if sha256_file(src) != sha256_file(dst):
                raise IOError(f"Drive copy checksum mismatch: {src.name}")
        os.rename(pending, target_dir)
    except Exception:
        log(f"Upload staging left for inspection: {pending}")
        raise


def finish_check(no):
    """Idempotent resume, ONLY after checksum/ZIP/QA/readback verification."""
    from src.m2_parsing.quality_checks import verify_tables_zip
    path = OUT / f"range_{no}"
    if not path.exists():
        return False
    files = [path / f"M2_range_{no}_tables.zip", path / "metrics_m2.json",
             path / "quality_report.json", path / "benchmark_summary.json",
             path / "triage_summary.json"]
    if not all(p.is_file() for p in files):
        raise ValueError(f"Range {no} incomplete: {path}")
    qa = json.loads(files[2].read_text(encoding="utf-8"))
    bench = json.loads(files[3].read_text(encoding="utf-8"))
    if not qa.get("passed") or qa.get("pipeline_revision") != 10:
        raise ValueError(f"Range {no} did not pass revision 10 QA.")
    if sha256_file(files[0]) != bench.get("zip_sha256"):
        raise ValueError(f"Range {no}: ZIP SHA256 differs from benchmark.")
    verify_tables_zip(files[0])
    from src.m2_parsing.selective_repair import read_metrics
    assert read_metrics(files[0]) == json.loads(files[1].read_text(encoding="utf-8"))
    log(f"Range {no} already exists, QA/SHA/CRC verified: SKIP")
    return True


def review_jsonl(metrics, dest):
    from src.m2_parsing.quality_checks import _semantic_review
    reviews, _ = _semantic_review(metrics["coverage"])
    with Path(dest).open("w", encoding="utf-8") as output:
        for row in reviews:
            output.write(json.dumps(row, ensure_ascii=False) + "\n")


## 8. Selective repair + merge range 1–2
Giữ nguyên paper nếu bản mới không chứng minh cải thiện; kiểm tra relationship và QA sau merge.


In [ ]:
def run_selective(service, no, m1_ranges, v6_ranges):
    from src.m2_parsing.selective_repair import (
        read_metrics, select_repair_ids, rebuild_selected_from_checkpoints,
        merge_verified_publications, incremental_quality_report)
    from src.m2_parsing.quality_checks import (
        validate_range_output, verify_tables_zip, _semantic_review)
    label = f"range_{no}"
    work = LOCAL / label
    olddir = work / "baseline"
    olddir.mkdir(parents=True, exist_ok=True)
    remote_files = {x["name"]: x for x in list_children(service, v6_ranges[label])}
    wanted = [f"M2_{label}_tables.zip", "quality_report.json",
              "metrics_m2.json", "benchmark_summary.json"]
    for name in wanted:
        if name not in remote_files:
            raise FileNotFoundError(f"Missing v6/{label}/{name}")
        download(service, remote_files[name], olddir / name)
    old_zip = olddir / wanted[0]
    metrics = read_metrics(old_zip)
    qa_old = json.loads((olddir / "quality_report.json").read_text(encoding="utf-8"))
    bench_old = json.loads((olddir / "benchmark_summary.json").read_text(encoding="utf-8"))
    assert metrics["pipeline_revision"] == 10 and qa_old["passed"] is True
    assert metrics == json.loads((olddir / "metrics_m2.json").read_text(encoding="utf-8"))
    assert sha256_file(old_zip) == bench_old["zip_sha256"]
    selected = select_repair_ids(metrics, MODE)
    log(f"Repair {label}: {len(selected)} selected publications from v6.")
    freshzip, report, qa, final_metrics = old_zip, None, qa_old, metrics
    changed = False
    if selected:
        checkpoint = work / "selected_checkpoints"
        checkpoint.mkdir(parents=True, exist_ok=True)
        remote_m1 = {x["name"]: x for x in list_children(service, m1_ranges[label + "_dev"])}
        discovered = set()
        archives = []
        for obj in sorted(remote_m1.values(), key=lambda o: o["name"]):
            if not (obj["name"].startswith("batch_") and obj["name"].endswith(".json")):
                continue
            manifest_file = work / "manifests" / obj["name"]
            download(service, obj, manifest_file)
            doc = json.loads(manifest_file.read_text(encoding="utf-8"))
            targets = selected.intersection(doc["ids"])
            if targets:
                tar_name = obj["name"][:-5] + ".tar.gz"
                if tar_name not in remote_m1:
                    raise FileNotFoundError(f"Missing {tar_name} for selected IDs.")
                if discovered.intersection(targets):
                    raise ValueError("Duplicate selected IDs across M1 manifests.")
                discovered.update(targets)
                shutil.copy2(manifest_file, checkpoint / obj["name"])
                archives.append(remote_m1[tar_name])
        if discovered != selected:
            raise ValueError(f"{label}: Missing {sorted(selected - discovered)[:15]} in M1.")
        for i, obj in enumerate(archives, 1):
            download(service, obj, checkpoint / obj["name"])
            if i % 10 == 0 or i == len(archives):
                log(f"Repair {label}: downloaded {i}/{len(archives)} selected M1 archives")
        retry = work / "retry"
        rebuild_selected_from_checkpoints(checkpoint, selected, work / "working", retry)
        proposed = work / f"M2_{label}_tables.zip"
        report = merge_verified_publications(old_zip, retry, proposed, expected_ids=selected)
        changed = bool(report["changed"])
        if changed:
            freshzip = proposed
            final_metrics = read_metrics(freshzip)
            if FULL_QA_REPAIR:
                extracted = work / "full_qa"
                extracted.mkdir(parents=True, exist_ok=True)
                with zipfile.ZipFile(freshzip) as z:
                    z.extractall(extracted)
                qa = validate_range_output(extracted, check_rows=True)
                qa.update({"validation_mode": "FULL_RESCAN",
                           "full_qa_rerun": True,
                           "base_pipeline_revision": 10})
                shutil.rmtree(extracted)
            else:
                qa, _ = incremental_quality_report(
                    freshzip, qa_old, original_revision=10)
            if not qa["passed"]:
                raise RuntimeError(f"{label}: merged QA failed; v6 untouched")
            log(f"Repair {label}: accepted {len(report['accepted'])} / {len(selected)}.")
        else:
            log(f"Repair {label}: no proven structural improvement; keeping v6 unchanged.")
    publish = work / "publish"
    publish.mkdir(exist_ok=True)
    zipdst = publish / f"M2_{label}_tables.zip"
    shutil.copy2(freshzip, zipdst)
    write_json(publish / "metrics_m2.json", final_metrics)
    write_json(publish / "quality_report.json", qa)
    if report is None:
        report = {"changed": False, "selected_count": len(selected),
                  "accepted": {}, "rejected": {}, "note": "No selected publication."}
    write_json(publish / "selective_merge_report.json", report)
    review_jsonl(final_metrics, publish / "semantic_review.jsonl")
    bench = dict(bench_old)
    bench.update({"range": label, "pipeline_revision": 10,
                  "base_pipeline_revision": 10, "zip_sha256": sha256_file(zipdst),
                  "tables_zip_bytes": zipdst.stat().st_size,
                  "n_publications": final_metrics["n_publications"],
                  "n_versions_total": final_metrics["n_versions_total"],
                  "n_versions_parsed": final_metrics["n_versions_parsed"],
                  "n_versions_empty": final_metrics["n_versions_empty"],
                  "n_versions_partial": final_metrics["n_versions_partial"],
                  "n_versions_failed": final_metrics["n_versions_failed"],
                  "run_utc": now(), "selective_accepted_count": len(report["accepted"]),
                  "selective_rejected_count": len(report["rejected"]),
                  "runtime_note": "Conservative targeted patch of existing v6; no full 10k rerun"})
    write_json(publish / "benchmark_summary.json", bench)
    write_json(publish / "repair_engine_manifest.json",
               {"base": "M2_processed_v6", "code": CODE_ZIP.name,
                "patch": "M2_FINAL_WRAPPER_OVERLAY_20261115",
                "selected": len(selected), "accepted": len(report["accepted"]),
                "rejected": len(report["rejected"]),
                "full_qa_rerun": bool(changed and FULL_QA_REPAIR), "time": now()})
    triage = emit_triage(zipdst, publish)
    stage_publish(publish, OUT / label)
    log(f"FINISHED {label} copy-on-write. Triage: {triage}")
    shutil.rmtree(work, ignore_errors=True)


## 9. Chạy mới range 3–5
Tải checkpoint, parse, kiểm tra QA, ZIP, SHA và xuất report.


In [ ]:
def run_fresh(service, no, m1_ranges):
    from src.m2_parsing.batch_pipeline import discover_checkpoint_batches
    from src.m2_parsing.selective_repair import read_metrics
    from src.m2_parsing.quality_checks import validate_range_output, verify_tables_zip
    label = f"range_{no}"
    work = LOCAL / label
    batch_dir = work / "checkpoints"
    table_dir = work / "tables"
    working_dir = work / "working"
    batch_dir.mkdir(parents=True, exist_ok=True)
    remote = [f for f in list_children(service, m1_ranges[label + "_dev"])
              if f["name"].startswith("batch_")
              and f["name"].endswith((".tar.gz", ".json"))]
    names = [f["name"] for f in remote]
    if len(set(names)) != len(names):
        raise RuntimeError(f"{label}: duplicate checkpoint names.")
    tarballs = [f for f in remote if f["name"].endswith(".tar.gz")]
    assert tarballs, f"{label}: no M1 tar checkpoints."
    for tar in tarballs:
        if tar["name"][:-7] + ".json" not in names:
            raise FileNotFoundError(f"{label}: missing manifest for {tar['name']}")
    log(f"{label}: downloading {len(remote)} M1 archive/manifest files.")
    for idx, obj in enumerate(sorted(remote, key=lambda x: x["name"]), 1):
        download(service, obj, batch_dir / obj["name"])
        if idx % 20 == 0 or idx == len(remote):
            log(f"{label}: downloaded {idx}/{len(remote)} files.")
    batches = discover_checkpoint_batches(batch_dir)
    expected = sum(len(x[2]["ids"]) for x in batches)
    log(f"{label}: {len(batches)} manifest batches, {expected} publications.")
    command = [sys.executable, "-u", "-m", "src.m2_parsing.run_m2",
               "--batch-dir", str(batch_dir), "--working-dir", str(working_dir),
               "--tables-dir", str(table_dir), "--workers", str(WORKERS)]
    rc = subprocess.run(command, cwd=PROJECT, check=False).returncode
    if rc not in (0, 2):
        raise RuntimeError(f"{label}: parser failed with code {rc}. No Drive output created.")
    metrics = read_metrics(table_dir / "metrics_m2.json")
    if metrics["pipeline_revision"] != 10 or metrics["n_publications"] != expected:
        raise ValueError(f"{label}: output does not match checkpoint coverage/revision.")
    qa = validate_range_output(table_dir, batch_dir, check_rows=FULL_QA_NEW)
    if not qa["passed"]:
        raise ValueError(f"{label}: quality check failed.")
    localzip = work / f"M2_{label}_tables.zip"
    members = ["elements.jsonl", "hierarchy.jsonl", "bibtex.jsonl", "metrics_m2.json"]
    with zipfile.ZipFile(localzip, "w", zipfile.ZIP_DEFLATED, compresslevel=1,
                         allowZip64=True) as z:
        for name in members:
            z.write(table_dir / name, arcname=name)
    verify_tables_zip(localzip)
    zip_sha = sha256_file(localzip)
    pub = work / "publish"
    pub.mkdir(exist_ok=True)
    shutil.copy2(localzip, pub / localzip.name)
    shutil.copy2(table_dir / "metrics_m2.json", pub / "metrics_m2.json")
    write_json(pub / "quality_report.json", qa)
    review_jsonl(metrics, pub / "semantic_review.jsonl")
    bench = {"range": label, "pipeline_revision": 10, "colab_hardware": "CPU-only",
             "workers": WORKERS, "n_publications": expected,
             "n_versions_total": metrics["n_versions_total"],
             "n_versions_parsed": metrics["n_versions_parsed"],
             "n_versions_empty": metrics["n_versions_empty"],
             "n_versions_partial": metrics["n_versions_partial"],
             "n_versions_failed": metrics["n_versions_failed"],
             "tables_zip_bytes": localzip.stat().st_size,
             "zip_sha256": zip_sha, "run_utc": now(),
             "runtime_note": "Fresh M2 parse with conservative final wrapper overlay."}
    write_json(pub / "benchmark_summary.json", bench)
    write_json(pub / "repair_engine_manifest.json",
               {"source": "M1 checkpoints", "code": CODE_ZIP.name,
                "patch": "M2_FINAL_WRAPPER_OVERLAY_20261115",
                "full_qa_rerun": FULL_QA_NEW, "time": now()})
    triage = emit_triage(localzip, pub)
    stage_publish(pub, OUT / label)
    log(f"FINISHED {label} fresh parse. Triage: {triage}")
    shutil.rmtree(work, ignore_errors=True)


## 10. Tạo master và summary


In [ ]:
def build_master():
    if not CREATE_MASTER:
        return
    if not all((OUT / f"range_{i}" / "quality_report.json").exists()
               for i in range(1, 6)):
        log("Master deferred: not all five ranges are available.")
        return
    from src.m2_parsing.quality_checks import combine_verified_range_zips, verify_tables_zip
    master = OUT / "M2_master_PROVISIONAL_tables.zip"
    if master.exists():
        log("Existing PROVISIONAL master preserved. Delete manually if rebuilding is desired.")
        return
    pending = OUT / ".M2_master_PROVISIONAL_incomplete.zip"
    if pending.exists():
        raise FileExistsError(f"Interrupted master: {pending}")
    log("Combining five QA-verified ranges into PROVISIONAL master...")
    manifest = combine_verified_range_zips(OUT, [1, 2, 3, 4, 5], pending)
    with zipfile.ZipFile(pending) as master_zip:
        broken_member = master_zip.testzip()
        if broken_member:
            raise IOError(f"Master ZIP failed CRC: {broken_member}")
    os.replace(pending, master)
    manifest["validation_label"] = "PROVISIONAL_SEMANTIC_REVIEW_PENDING"
    manifest["sha256"] = sha256_file(master)
    manifest["created_utc"] = now()
    write_json(OUT / "M2_master_manifest.json", manifest)
    log(f"Master created: {master}")


def final_summary():
    rows = []
    for i in range(1, 6):
        path = OUT / f"range_{i}"
        qa_file = path / "quality_report.json"
        if not qa_file.exists():
            rows.append({"range": i, "status": "MISSING"})
            continue
        qa = json.loads(qa_file.read_text(encoding="utf-8"))
        triage = json.loads((path / "triage_summary.json").read_text(encoding="utf-8"))
        metrics = json.loads((path / "metrics_m2.json").read_text(encoding="utf-8"))
        rows.append({"range": i, "qa_passed": qa.get("passed"),
                     "n_publications": metrics["n_publications"],
                     "parsed_versions": metrics["n_versions_parsed"],
                     "partial_versions": metrics["n_versions_partial"],
                     "failed_versions": metrics["n_versions_failed"],
                     "source_heading_review": qa.get("source_heading_review_count"),
                     "semantic_review": qa.get("semantic_review_count"),
                     "triage": triage})
    write_json(OUT / "ONEPASS_FINAL_SUMMARY.json",
               {"created_utc": now(), "parser_revision": 10,
                "overlay": "M2_FINAL_WRAPPER_OVERLAY_20261115",
                "dataset_certification": "PROVISIONAL until high-risk semantic cases reviewed",
                "ranges": rows})
    log("FINAL SUMMARY")
    for row in rows:
        log(json.dumps({k: v for k, v in row.items() if k != "triage"}, ensure_ascii=False))
    log(f"Output folder: {OUT}")


## 11. Preflight: test trước khi xử lý dữ liệu
Chạy đủ test cũ và test patch; lỗi thì dừng.


In [ ]:
preflight()
_service=drive_client()
def _find_by_name(name):
    safe=name.replace("'", "\\'")
    q=f"name = '{safe}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
    result=_service.files().list(q=q,fields="nextPageToken,files(id,name,parents)",pageSize=100,supportsAllDrives=True,includeItemsFromAllDrives=True).execute()
    return result.get("files",[])
if M1_PARENT_ID=="AUTO":
    candidates=_find_by_name("range_1_dev")
    parents=[x["parents"][0] for x in candidates if x.get("parents")]
    M1_PARENT_ID=parents[0] if len(set(parents))==1 else input("Paste M1 checkpoints parent folder ID: ").strip()
if V6_FOLDER_ID=="AUTO":
    candidates=_find_by_name("M2_processed_v6")
    V6_FOLDER_ID=candidates[0]["id"] if len(candidates)==1 else input("Paste M2_processed_v6 folder ID: ").strip()
assert M1_PARENT_ID and M1_PARENT_ID!="AUTO"
assert V6_FOLDER_ID and V6_FOLDER_ID!="AUTO"
print("Found M1 checkpoints / v6 folders.")
_m1=folder_map(_service,M1_PARENT_ID)
_v6=folder_map(_service,V6_FOLDER_ID)
for _r in REPAIR_RANGES+NEW_RANGES:
    assert f'range_{_r}_dev' in _m1, f'M1 missing range_{_r}_dev'
for _r in REPAIR_RANGES:
    assert f'range_{_r}' in _v6, f'v6 missing range_{_r}'
print('PREFLIGHT PASS')


## 12. Chạy repair range 1–2 và gộp tự động


In [ ]:
for _r in REPAIR_RANGES:
    print(f'========== REPAIR range_{_r} ==========',flush=True)
    if SKIP_VERIFIED_FINISHED and finish_check(_r): continue
    run_selective(_service,_r,_m1,_v6)
    assert finish_check(_r)
print('Repair completed / verified.')


## 13. Chạy range 3–5
Có thể chia làm nhiều lần. Range đạt QA/CRC/SHA trước đó sẽ được bỏ qua.


In [ ]:
for _r in NEW_RANGES:
    print(f'========== NEW range_{_r} ==========',flush=True)
    if SKIP_VERIFIED_FINISHED and finish_check(_r): continue
    run_fresh(_service,_r,_m1)
    assert finish_check(_r)
print('New ranges completed / verified.')


## 14. Ghép master PROVISIONAL và summary


In [ ]:
build_master()
final_summary()
print('OUTPUT:',OUT)


## 15. Xem QA, partial/failed và semantic review
QA PASS không chứng minh mọi bài được parse hoàn hảo. Các cảnh báo nghiêm trọng vẫn được giữ để review.


In [ ]:
import pandas as pd
rows=[]
for i in range(1,6):
    d=OUT/f'range_{i}'
    if not (d/'quality_report.json').is_file():
        rows.append({'range':i,'status':'MISSING'});continue
    q=json.loads((d/'quality_report.json').read_text(encoding='utf-8'))
    m=json.loads((d/'metrics_m2.json').read_text(encoding='utf-8'))
    t=json.loads((d/'triage_summary.json').read_text(encoding='utf-8'))
    rows.append({'range':i,'qa':q.get('passed'),'papers':m.get('n_publications'),'parsed':m.get('n_versions_parsed'),'partial':m.get('n_versions_partial'),'failed':m.get('n_versions_failed'),'semantic_review':q.get('semantic_review_count'),'high_heading':t.get('HIGH_HEADING_REVIEW',0)})
display(pd.DataFrame(rows))
